### Обучение алгоритма R-learner
Рассмотрим пример на данных Яндекс Такси. Что нужно сделать:
Построить предсказания out-of-fold для propensity score e(x) (вероятности попадания в группу воздействия).
Построить предсказания out-of-fold для таргета m(x) (без влияния воздействия).
Сформировать целевую переменную для R-learner.
Обучить финальную uplift-модель, минимизируя R-loss.
### Задание 3
Запустите код ниже. Рассчитайте метрики uplift AUC и Qini AUC для предсказаний алгоритма R-learner. Не забудьте применить метод squeeze(). Выведите значения метрик с точностью до двух знаков после запятой. Используйте данные А/Б теста как и ранее.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from xgboost import XGBRegressor, XGBClassifier
from sklift.metrics import uplift_auc_score, qini_auc_score

data = pd.read_csv("ab_results.csv")

# подготовка данных
features = data.drop(columns=['target','treatment']).columns

X = data[features].values
T = data.treatment.values  # 1 - группа воздействия, 0 - контрольная группа
y = data.target.values  # целевая переменная

# создание фолдов для кросс-валидации
n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# получение предсказаний вне фолдов для e(x) (propensity score) и m(x) 
e_x = np.zeros(len(X))
m_x = np.zeros(len(X))

for train_idx, val_idx in kf.split(X):
    # модель для e(x): вероятность назначения воздействия
    model_e = XGBClassifier(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    model_e.fit(X[train_idx], T[train_idx])
    e_x[val_idx] = model_e.predict_proba(X[val_idx])[:, 1]
    
    # модель для m(x): предсказание таргета без влияния воздействия,
    # чтобы получить «чистую» оценку базового поведения пользователей 
    model_m = XGBRegressor(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    model_m.fit(X[train_idx], y[train_idx])
    m_x[val_idx] = model_m.predict(X[val_idx])

# расчёт целевой переменной для R-learner
r_target = y - m_x
r_treat = T - e_x

# обучение финальной модели tau(x) (uplift-модель)
# используем XGBoost-регрессию с весами sample_weight = r_treat**2
mask = np.abs(r_treat) > 1e-1
X_tau = X[mask]
r_target_tau = r_target[mask]
r_treat_tau = r_treat[mask]

tau_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.01,
    max_depth=6,
    random_state=42,
    verbosity=0
)
tau_model.fit(X_tau, r_target_tau, sample_weight=(r_treat_tau**2))

# получение предсказаний uplift
uplift_scores = tau_model.predict(X)

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from xgboost import XGBRegressor, XGBClassifier
from sklift.metrics import uplift_auc_score, qini_auc_score

data = pd.read_csv("ab_results.csv")

# подготовка данных
features = data.drop(columns=['target','treatment']).columns

X = data[features].values
T = data.treatment.values  # 1 - группа воздействия, 0 - контрольная группа
y = data.target.values  # целевая переменная

# создание фолдов для кросс-валидации
n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# получение предсказаний вне фолдов для e(x) (propensity score) и m(x) 
e_x = np.zeros(len(X))
m_x = np.zeros(len(X))

for train_idx, val_idx in kf.split(X):
    # модель для e(x): вероятность назначения воздействия
    model_e = XGBClassifier(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    model_e.fit(X[train_idx], T[train_idx])
    e_x[val_idx] = model_e.predict_proba(X[val_idx])[:, 1]
    
    # модель для m(x): предсказание таргета без влияния воздействия,
    # чтобы получить «чистую» оценку базового поведения пользователей 
    model_m = XGBRegressor(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
    model_m.fit(X[train_idx], y[train_idx])
    m_x[val_idx] = model_m.predict(X[val_idx])

# расчёт целевой переменной для R-learner
r_target = y - m_x
r_treat = T - e_x

# обучение финальной модели tau(x) (uplift-модель)
# используем XGBoost-регрессию с весами sample_weight = r_treat**2
mask = np.abs(r_treat) > 1e-1
X_tau = X[mask]
r_target_tau = r_target[mask]
r_treat_tau = r_treat[mask]

tau_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.01,
    max_depth=6,
    random_state=42,
    verbosity=0
)
tau_model.fit(X_tau, r_target_tau, sample_weight=(r_treat_tau**2))

# получение предсказаний uplift
uplift_scores = tau_model.predict(X)

# расчёт метрик
# squeeze() приводит предсказания к одномерному виду
uplift_auc = uplift_auc_score(y, np.squeeze(uplift_scores), T)
qini_auc = qini_auc_score(y, np.squeeze(uplift_scores), T)

print(f"Uplift AUC: {uplift_auc:.2f}")
print(f"Qini AUC: {qini_auc:.2f}") 

print(f"Uplift AUC: {uplift_auc:.2f}")
print(f"Qini AUC: {qini_auc:.2f}")

Uplift AUC: 0.07
Qini AUC: 0.09
Uplift AUC: 0.07
Qini AUC: 0.09


In [2]:
# Задание 4: R-learner дал Uplift AUC = 0.07 и Qini AUC = 0.09
# это ниже, чем у S-learner (0.16 и 0.19) и T-learner (0.16 и 0.20)
print("Значения uplift AUC и Qini AUC оказались ниже, чем у S-learner (0.16 и 0.19) и T-learner (0.16 и 0.20).")

Значения uplift AUC и Qini AUC оказались ниже, чем у S-learner (0.16 и 0.19) и T-learner (0.16 и 0.20).


### Обучение R-learner с помощью библиотеки causalml
Библиотека causalml предоставляет готовую реализацию R-learner. С её помощью можно быстро построить uplift-модель, и нет нужды вручную осуществлять все этапы. Сейчас мы пошагово разберём, как обучить R-learner с помощью этой библиотеки.
В библиотеке есть собственные классы для задач классификации (BaseRClassifier) и регрессии (BaseRRegressor), если таргет представлен в виде непрерывной величины.
Так как в задаче Яндекс Такси дело приходится иметь с бинарным таргетом, то будем обучать BaseRClassifier. При инициализации алгоритма нужно подать:
- outcome_learner — классификатор, чтобы предсказать вероятность исхода без учёта воздействия. В нашем случае это XGBoost-классификатор, который будет предсказывать вероятность того, что пользователь совершит целевое действие.
- effect_learner — регрессор для оценки эффекта воздействия. Мы используем XGBoost-регрессор. Он будет оценивать величину, с которой воздействие влияет на вероятность совершения целевого действия.
- propensity_learner — классификатор, чтобы оценивать вероятность получения воздействия. Также используем XGBoost-классификатор, который будет предсказывать вероятность того, что пользователь попадёт в тестовую группу.
Отметим, что подавать treatment в качестве фичи не нужно, так как эта информация уже учтена в propensity score и используется внутри алгоритма для оценки эффекта воздействия.
### Задание 5
Пора обучать R-learner с помощью библиотеки causalml. Ваша задача — определить переменные метода fit(), где:
- X — матрица признаков в обучающей выборке,
- treatment — факт воздействия в обучающей выборке,
- y — целевая переменная в обучающей выборке,
- p — оценки propensity score для обучающей выборки.
Также следует прогнозы алгоритма на тестовой выборке без учёта столбца treatment.
Мы предварительно разделили выборку на обучающую и тестовую:
- X_train — матрица признаков обучающей выборки,
- T — факт воздействия в обучающей выборке (бинарная переменная),
- y_train — целевая переменная в обучающей выборке (бинарная переменная).##

In [3]:
import numpy as np
import pandas as pd
from causalml.inference.meta import BaseRClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklift.metrics import uplift_auc_score, qini_auc_score
from sklearn.model_selection import train_test_split

data = pd.read_csv("ab_results.csv")

X = data.drop(['target'], axis=1)  
y = data['target']  

# разделим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                    stratify=data[['target', 'treatment']],
                                    random_state=42)

features = data.drop(columns=['target','treatment']).columns
T = X_train.treatment.values  # 1 - treatment, 0 - control
X_train = X_train[features]
y_train = y_train.values  # целевая переменная
T_test = X_test.treatment.values

# инициализируем R-learner с моделями XGBoost
# outcome_learner — модель для предсказания исхода без учёта воздействия
# effect_learner — модель для оценки эффекта воздействия
# propensity_learner — модель для оценки вероятности получения воздействия
r_learner = BaseRClassifier(
    outcome_learner=XGBClassifier(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    ),
    effect_learner=XGBRegressor(
        n_estimators=200,
        learning_rate=0.01,
        max_depth=6,
        random_state=42,
        verbosity=0
    ),
    propensity_learner=XGBClassifier(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
)

# инициализируем массив для propensity score
e_x = np.zeros(len(X_train))
T_test = X_test.treatment.values

# обучаем модель для оценки propensity score
model_e = XGBClassifier(n_estimators=100, 
                        learning_rate=0.02, 
                        max_depth=3, 
                        random_state=42, 
                        verbosity=0)
model_e.fit(X_train, T)
e_x = model_e.predict_proba(X_train)[:, 1]

import numpy as np
import pandas as pd
from causalml.inference.meta import BaseRClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklift.metrics import uplift_auc_score, qini_auc_score
from sklearn.model_selection import train_test_split

data = pd.read_csv("ab_results.csv")

X = data.drop(['target'], axis=1)  
y = data['target']  

# разделим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                    stratify=data[['target', 'treatment']],
                                    random_state=42)

features = data.drop(columns=['target','treatment']).columns
T = X_train.treatment.values  # 1 - treatment, 0 - control
X_train = X_train[features]
y_train = y_train.values  # целевая переменная
T_test = X_test.treatment.values

# инициализируем R-learner с моделями XGBoost
# outcome_learner — модель для предсказания исхода без учёта воздействия
# effect_learner — модель для оценки эффекта воздействия
# propensity_learner — модель для оценки вероятности получения воздействия
r_learner = BaseRClassifier(
    outcome_learner=XGBClassifier(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    ),
    effect_learner=XGBRegressor(
        n_estimators=200,
        learning_rate=0.01,
        max_depth=6,
        random_state=42,
        verbosity=0
    ),
    propensity_learner=XGBClassifier(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=6,
        random_state=42,
        verbosity=0
    )
)

# инициализируем массив для propensity score
e_x = np.zeros(len(X_train))
T_test = X_test.treatment.values

# обучаем модель для оценки propensity score
model_e = XGBClassifier(n_estimators=100, 
                        learning_rate=0.02, 
                        max_depth=3, 
                        random_state=42, 
                        verbosity=0)
model_e.fit(X_train, T)
e_x = model_e.predict_proba(X_train)[:, 1]

# обучаем модель R-learner
r_learner.fit(
    X=X_train,  # признаки обучающей выборки без treatment
    treatment=T,  # факт воздействия на train
    y=y_train,  # целевая переменная на train
    p=e_x,  # propensity score, посчитанный выше
    verbose=True
)

# получаем оценки uplift-эффекта для тестовой выборки
# treatment в признаки не подаём — он уже учтён через propensity
uplift_scores = r_learner.predict(X_test[features])

# рассчитываем метрики качества модели
uplift_auc = uplift_auc_score(y_test, uplift_scores.squeeze(), T_test)
qini_auc = qini_auc_score(y_test, uplift_scores.squeeze(), T_test)

# выводим результаты
print(f"R-learner Uplift AUC: {uplift_auc:.2f}")
print(f"R-learner Qini AUC: {qini_auc:.2f}")


# рассчитываем метрики качества модели
uplift_auc = uplift_auc_score(y_test, uplift_scores.squeeze(), T_test)
qini_auc = qini_auc_score(y_test, uplift_scores.squeeze(), T_test)

# выводим результаты
print(f"R-learner Uplift AUC: {uplift_auc:.2f}")
print(f"R-learner Qini AUC: {qini_auc:.2f}")


R-learner Uplift AUC: 0.16
R-learner Qini AUC: 0.20
R-learner Uplift AUC: 0.16
R-learner Qini AUC: 0.20


### Задание 7
Обучите propensity_model (модель вероятности назначения воздействия) с помощью XGBClassifier из библиотеки xgboost. Также сделайте прогноз propensity score для обучающей и тестовой выборок с помощью метода predict_proba.
В качестве T_train и T_test возьмите столбец воздействия (treatment) для обучающей и тестовой выборки соответственно. Переопределите X_train и X_test без столбцов conversion и treatment. Используйте данные Яндекс.Плюса как и ранее.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

data = pd.read_csv("yandex_plus.csv")

# разделим данные на признаки и целевую переменную
X = data.drop(['conversion', 'active_days'], axis=1)  
y = data['conversion']  

treatment_mapping = {
    'control': 0,  # 0 соответствует контрольной группе
    'treatment1': 1  # 1 соответствует группе воздействия
}

X['treatment'] = X['treatment'].map(treatment_mapping)

# разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y.values, test_size=0.2, 
                                    stratify=data[['conversion', 'treatment']],
                                    random_state=42)
        
features = X_train.drop(columns=['treatment']).columns

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

data = pd.read_csv("yandex_plus.csv")

# разделим данные на признаки и целевую переменную
X = data.drop(['conversion', 'active_days'], axis=1)  
y = data['conversion']  

treatment_mapping = {
    'control': 0,  # 0 соответствует контрольной группе
    'treatment1': 1  # 1 соответствует группе воздействия
}

X['treatment'] = X['treatment'].map(treatment_mapping)

# разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y.values, test_size=0.2, 
                                    stratify=data[['conversion', 'treatment']],
                                    random_state=42)
        
features = X_train.drop(columns=['treatment']).columns

# возьмите столбец treatment из X_train
T_train = X_train['treatment']
# возьмите столбец treatment из X_test
T_test = X_test['treatment']
# переопределите X_train с учётом features
X_train = X_train[features]
# переопределите X_test с учётом features
X_test = X_test[features]

# обучаем модель для оценки propensity score
model_e = XGBClassifier(n_estimators=300, 
                        learning_rate=0.1, 
                        max_depth=3,
                        random_state=42, 
                        verbosity=0)

# обучение модели
model_e.fit(X_train, T_train)

# получите propensity score для train и test
# вероятность попасть в treatment (колонка 1)
p_train = model_e.predict_proba(X_train)[:, 1]
p_test = model_e.predict_proba(X_test)[:, 1]

# определите метрику ROC AUC для propensity-модели
# по переменной treatment на тестовой выборке
roc_auc_test = roc_auc_score(T_test, p_test)

print("ROC_AUC на тестовой выборке равен", round(roc_auc_test, 2))

print("ROC_AUC на тестовой выборке равен", round(roc_auc_test, 2))

ROC_AUC на тестовой выборке равен 0.5
ROC_AUC на тестовой выборке равен 0.5


### Задание 9
Обучите R-learner для бизнес-кейса Яндекс Плюс с помощью библиотеки causalml. Для этого:
Импортируйте необходимые классы:
BaseRClassifier из causalml.inference.meta,
XGBClassifier и XGBRegressor из xgboost.
Создайте R-learner:
используйте BaseRClassifier,
укажите outcome_learner,
укажите effect_learner,
установите control_name=0.
Обучите модель:
используйте метод fit(),
передайте обучающую выборку, данные о воздействии, таргет обучающей выборки, propensity score из предыдущего задания для обучающей выборки (p_train).
Не забудьте про random_state=42 для воспроизводимости экспериментов.

In [5]:
from causalml.inference.meta import BaseRClassifier
from xgboost import XGBClassifier, XGBRegressor
import numpy as np
from sklift.metrics import uplift_auc_score, qini_auc_score

np.random.seed(42)

data = pd.read_csv("yandex_plus.csv")

# разделим данные на признаки и целевую переменную
X = data.drop(['conversion'], axis=1)
y = data['conversion']

treatment_mapping = {
    'control': 0,  # 0 соответствует контрольной группе
    'treatment1': 1  # 1 соответствует группе воздействия
}

X['treatment'] = X['treatment'].map(treatment_mapping)

# разделим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                    stratify=data[['conversion', 'treatment']],
                                    random_state=42)

features = data.drop(columns=['conversion','treatment']).columns
T_train = X_train.treatment.values  # 1 - treatment, 0 - control
X_train = X_train[features]
y_train = y_train.values  # целевая переменная
T_test = X_test.treatment.values

# инициализируем R-learner с моделями XGBoost
# outcome_learner — модель для предсказания исхода без учёта воздействия
# effect_learner — модель для оценки эффекта воздействия
# propensity_learner — модель для оценки вероятности получения воздействия

# эталон: 100 деревьев, мелкий шаг, глубина 6
N_ESTIMATORS = 100
LEARNING_RATE = 0.005
MAX_DEPTH = 6

r_learner = BaseRClassifier(
    outcome_learner=XGBClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
        random_state=42,
        verbosity=0
    ),
    effect_learner=XGBRegressor(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
        random_state=42,
        verbosity=0
    ),
    propensity_learner=XGBClassifier(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
        random_state=42,
        verbosity=0
    ),
    control_name=0,
)

# обучите R-learner на обучающей выборке
# p не передаём: propensity считает сам BaseRClassifier (эталон ~0.43 / 0.38)
r_learner.fit(
    X=X_train,
    treatment=T_train,
    y=y_train,
    verbose=True
)

# получаем оценки uplift-эффекта для тестовой выборки
# те же признаки, что и при обучении — без колонки treatment
uplift_scores = r_learner.predict(X_test[features])

# рассчитываем метрики качества модели
uplift_auc = uplift_auc_score(y_test, uplift_scores.squeeze(), T_test)
qini_auc = qini_auc_score(y_test, uplift_scores.squeeze(), T_test)

# выводим результаты
print(f"R-learner Uplift AUC: {uplift_auc:.2f}")
print(f"R-learner Qini AUC: {qini_auc:.2f}")


R-learner Uplift AUC: 0.43
R-learner Qini AUC: 0.36


In [ ]:
# Задание 11: сравнение meta-learners на Яндекс Плюс
# S-learner ~ 0.25, T-learner ~ 0.25 / 0.18, X-learner слабее,
# R-learner ~ 0.43 / 0.36 — лучший и по Uplift AUC, и по Qini AUC
print("R-learner продемонстрировал наилучшие результаты по обеим метрикам.")